# GPU Budget Negotiation Arena · End-to-end Training Notebook

This is the single judge-facing notebook. Running every cell from the top in a Colab T4 runtime produces:

1. A clean repo checkout, dependencies, and the unit-test smoke pass.
2. All small artifacts (`baseline_eval.json`, `holdout_eval.json`, `training_curve.json`, the SFT data, the demo + judged transcripts).
3. A real Llama-3.2-3B SFT warm-start LoRA (`artifacts/sft-checkpoint`) and its loss curve.
4. The **trained LLM as a live policy** evaluated on every task and seed, scored against every scripted baseline. Output: `artifacts/trained_llm_eval.json` and the headline `plots/trained_llm_vs_baselines.svg`.
5. A real **GRPO loop against the live environment reward**, starting from the SFT'd LoRA. Output: `artifacts/grpo-checkpoint`, `artifacts/grpo_training_curve.json`, `plots/grpo_reward_curve.svg`.
6. A second trained-LLM evaluation, this time with the GRPO'd weights, producing the trained-LLM-vs-baselines bar chart with two trained columns.
7. Everything copied to Drive and (optionally) pushed back to GitHub so the front end and README pick it up automatically.

CPU runtimes are safe — every GPU step prints `{"status":"skipped","reason":...}` and the rest of the notebook still runs end-to-end.


## 1. Mount Google Drive and clone

Use Drive for generated artifacts and checkpoints. Use GitHub for source code.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ['DRIVE_OUT'] = '/content/drive/MyDrive/gpu_budget_negotiation_arena'
os.makedirs(os.environ['DRIVE_OUT'], exist_ok=True)


In [ ]:
%cd /content
REPO_URL = "https://github.com/abhinavgautam01/GPU_Budget_Negotiation_Arena.git"
REPO_DIR = "/content/GPU_Budget_Negotiation_Arena"
BRANCH = "main"

import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    status = subprocess.run(["git", "-C", REPO_DIR, "status", "--porcelain"], check=True, capture_output=True, text=True).stdout.strip()
    if status:
        subprocess.run(["git", "-C", REPO_DIR, "stash", "push", "-u", "-m", "colab-autostash-before-sync"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

%cd /content/GPU_Budget_Negotiation_Arena
!git log -1 --oneline
!python -m pip install -q -e ".[dev]"
!python -m pytest -q


## 2. Generate required local artifacts

This produces baseline eval, reward-loop comparison, SFT traces, SFT chat messages, plots, and a demo transcript.


In [ ]:
%cd /content/GPU_Budget_Negotiation_Arena
!python scripts/check_submission.py
!mkdir -p "$DRIVE_OUT"
!cp -r artifacts plots data "$DRIVE_OUT"/
!ls -lh "$DRIVE_OUT"/artifacts


## 3. Optional Hugging Face login

Use this only if you want to push a model checkpoint or dataset from Colab. Do not paste tokens into committed files.


In [ ]:
# Uncomment when needed.
# !python -m pip install -q huggingface_hub
# from huggingface_hub import notebook_login
# notebook_login()


## 4. SFT warm start with Unsloth (GPU)

Run this on a GPU runtime. The SFT stage teaches JSON action formatting and basic negotiation behavior before GRPO. We bump `--max-steps` to **500** so you get the full 13-epoch loss curve already captured in `artifacts/sft_training_curve.json`. Lower it to ~120 for a quick smoke test.


In [ ]:
# GPU-only SFT warm start. Safe to run on CPU: it will skip instead of crashing.
%cd /content/GPU_Budget_Negotiation_Arena
import torch

if not torch.cuda.is_available():
    print({
        'status': 'skipped',
        'reason': 'No CUDA GPU detected. In Colab choose Runtime -> Change runtime type -> T4 GPU, then rerun from setup.',
        'torch': torch.__version__,
    })
else:
    !python -m pip install -q unsloth trl transformers datasets accelerate peft bitsandbytes
    !python training/run_unsloth_sft.py \
      --model-name unsloth/Llama-3.2-3B-Instruct \
      --data data/sft_messages.jsonl \
      --output artifacts/sft-checkpoint \
      --drive-output "$DRIVE_OUT" \
      --max-steps 500
    !python scripts/extract_sft_curve.py
    !ls -lh artifacts/sft-checkpoint artifacts/sft_training_curve.json plots/sft_loss_curve.svg


## 5. Trained LLM vs scripted baselines (SFT, before GRPO)

Take the SFT'd LoRA, wrap it as a live policy in the env, and roll out full episodes on every task and seed. Compare the resulting reward against every scripted baseline plus the **untrained base Llama** so the front end can show a clean before/after column.


In [ ]:
%cd /content/GPU_Budget_Negotiation_Arena
import torch

if not torch.cuda.is_available():
    print({'status': 'skipped', 'reason': 'No CUDA GPU; trained-LLM rollout requires a T4 or better.'})
else:
    !python scripts/evaluate_trained_llm.py \
      --base-model unsloth/Llama-3.2-3B-Instruct \
      --model-path artifacts/sft-checkpoint \
      --policy-name trained_llm \
      --include-base-model \
      --seeds 5 \
      --tasks single_trade market_round coalition_market \
      --output artifacts/trained_llm_eval.json
    !python scripts/plot_trained_vs_baselines.py
    !ls -lh artifacts/trained_llm_eval.json artifacts/trained_llm_summary.json plots/trained_llm_vs_baselines.svg


## 6. GRPO against the live environment reward

This is the headline RL loop. We start from the SFT'd LoRA, build a dataset of replayed observations (rule-expert advances the env to round R, then the model is asked for its action), and run TRL's `GRPOTrainer` with a custom reward function that:

- parses each completion as a `GpuNegotiationAction`,
- steps the live env with that action,
- and returns `obs.reward + format_bonus` (or `parse_penalty` if the JSON is malformed).

The trainer ranks the N completions per prompt and updates the LoRA so that high-reward completions become more likely. This is real GRPO — no surrogate, no proxy reward.

200 steps × 4 generations on a T4 takes roughly 25–35 minutes. Drop `--max-steps` to 60 for a quick smoke run.

In [ ]:
%cd /content/GPU_Budget_Negotiation_Arena
import torch

if not torch.cuda.is_available():
    print({'status': 'skipped', 'reason': 'No CUDA GPU; GRPO requires a T4 or better.'})
else:
    !python -m pip install -q unsloth trl transformers datasets accelerate peft bitsandbytes
    !python training/run_grpo_against_env.py \
      --base-model unsloth/Llama-3.2-3B-Instruct \
      --sft-checkpoint artifacts/sft-checkpoint \
      --output artifacts/grpo-checkpoint \
      --drive-output "$DRIVE_OUT" \
      --max-steps 200 \
      --num-generations 4 \
      --seeds-per-task 8 \
      --rounds-per-seed 4 \
      --logging-steps 5
    !ls -lh artifacts/grpo-checkpoint artifacts/grpo_training_curve.json plots/grpo_reward_curve.svg

## 7. Trained LLM vs scripted baselines (after GRPO)

Re-run the trained-LLM evaluation against the GRPO'd checkpoint and replot. The bar chart now has two trained columns — SFT and GRPO — alongside every scripted baseline. This is the "reward improvement evidence" the README and front end render.

In [ ]:
%cd /content/GPU_Budget_Negotiation_Arena
import json
from pathlib import Path
import torch

if not torch.cuda.is_available():
    print({'status': 'skipped', 'reason': 'No CUDA GPU; GRPO eval requires a T4 or better.'})
else:
    !python scripts/evaluate_trained_llm.py \
      --base-model unsloth/Llama-3.2-3B-Instruct \
      --model-path artifacts/grpo-checkpoint \
      --policy-name trained_llm_grpo \
      --seeds 5 \
      --tasks single_trade market_round coalition_market \
      --output artifacts/trained_llm_grpo_eval.json

    sft_path = Path('artifacts/trained_llm_eval.json')
    grpo_path = Path('artifacts/trained_llm_grpo_eval.json')
    if sft_path.exists() and grpo_path.exists():
        sft_payload = json.loads(sft_path.read_text(encoding='utf-8'))
        grpo_payload = json.loads(grpo_path.read_text(encoding='utf-8'))
        for task, bucket in (grpo_payload.get('summary') or {}).items():
            sft_payload.setdefault('summary', {}).setdefault(task, {}).update(bucket)
        sft_payload.setdefault('episodes', []).extend(grpo_payload.get('episodes', []))
        sft_payload.setdefault('action_distribution', {}).update(grpo_payload.get('action_distribution', {}))
        Path('artifacts/trained_llm_eval.json').write_text(
            json.dumps(sft_payload, indent=2) + '\n', encoding='utf-8'
        )
    !python scripts/plot_trained_vs_baselines.py
    !ls -lh artifacts/trained_llm_eval.json artifacts/trained_llm_grpo_eval.json plots/trained_llm_vs_baselines.svg

## 8. Final submission handoff

Verify final artifacts, copy them to Drive, and optionally push regenerated compact artifacts to GitHub. Keep large checkpoints in Drive or a Hugging Face model repo.


In [ ]:
%cd /content/GPU_Budget_Negotiation_Arena
!ls -lh \
  artifacts/judged_transcript.md \
  artifacts/before_after_training.md \
  artifacts/holdout_eval.json \
  artifacts/training_eval.json \
  artifacts/training_curve.json \
  artifacts/training_report.md \
  artifacts/sft_training_curve.json \
  artifacts/trained_llm_eval.json \
  artifacts/trained_llm_grpo_eval.json \
  artifacts/trained_llm_summary.json \
  artifacts/grpo_training_curve.json \
  plots/baseline_rewards.svg \
  plots/sft_loss_curve.svg \
  plots/grpo_reward_curve.svg \
  plots/trained_llm_vs_baselines.svg \
  plots/reward_progress.json
!mkdir -p "$DRIVE_OUT"
!cp -r artifacts plots data "$DRIVE_OUT"/
!ls -lh "$DRIVE_OUT"/artifacts
!ls -lh "$DRIVE_OUT"/plots
!git status --short

# Optional: push regenerated compact artifacts back to GitHub from Colab.
# !git add \
#   artifacts/baseline_eval.json artifacts/holdout_eval.json \
#   artifacts/demo_transcript.md artifacts/judged_transcript.md \
#   artifacts/before_after_training.md \
#   artifacts/training_eval.json artifacts/training_curve.json artifacts/training_report.md \
#   artifacts/sft_training_curve.json artifacts/trained_llm_eval.json \
#   artifacts/trained_llm_grpo_eval.json artifacts/trained_llm_summary.json \
#   artifacts/grpo_training_curve.json \
#   plots/baseline_rewards.svg plots/sft_loss_curve.svg \
#   plots/grpo_reward_curve.svg plots/trained_llm_vs_baselines.svg \
#   plots/reward_progress.json \
#   data/sft_traces.jsonl data/sft_messages.jsonl
# !git commit -m "Regenerate final submission artifacts"
# !git push origin main


## 9. Deployment checks

After pushing to Hugging Face Spaces from your local machine, run these checks. If the Space is still building, wait a minute and rerun.


In [ ]:
!curl -sS https://abhinavgautam01-gpu-budget-negotiation-arena.hf.space/health || true
!curl -sS https://abhinavgautam01-gpu-budget-negotiation-arena.hf.space/tasks || true
